# CareerAI — Resume Analyzer
> AI-powered resume analysis using TF-IDF cosine similarity and keyword matching

**Author:** Your Name  
**Tech Stack:** Python · Scikit-learn · pdfplumber · Streamlit  

---

## Project Overview

CareerAI analyzes a resume against a target job role and returns:
- A **match score** (keyword coverage + TF-IDF semantic similarity)
- **Skill gap detection** (matched, missing, bonus skills)
- **Seniority detection** from resume language
- **Actionable suggestions** for improvement

## Notebook Structure
1. Install dependencies
2. Skills database
3. TF-IDF similarity engine
4. Resume analysis pipeline
5. Test with sample resumes
6. Visualize results

## 1. Install Dependencies

In [ ]:
!pip install scikit-learn pandas numpy matplotlib pdfplumber streamlit

## 2. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print('All libraries imported successfully!')

## 3. Skills Database
Define required and bonus skills for each supported job role.

In [ ]:
skills = {
    "software developer": {
        "required": ["python", "java", "c++", "data structures", "algorithms", "git", "sql", "api", "react"],
        "bonus": ["docker", "kubernetes", "aws", "typescript", "graphql", "microservices"]
    },
    "data analyst": {
        "required": ["python", "pandas", "numpy", "excel", "sql", "statistics", "power bi", "tableau"],
        "bonus": ["r", "spark", "airflow", "looker", "dbt", "bigquery"]
    },
    "machine learning engineer": {
        "required": ["python", "scikit-learn", "tensorflow", "pandas", "numpy", "data preprocessing", "model training", "pytorch"],
        "bonus": ["mlflow", "docker", "aws sagemaker", "feature engineering", "nlp", "deep learning"]
    },
    "frontend developer": {
        "required": ["html", "css", "javascript", "react", "typescript", "git", "responsive design"],
        "bonus": ["nextjs", "tailwind", "webpack", "testing", "accessibility", "figma"]
    },
    "backend developer": {
        "required": ["python", "java", "node.js", "sql", "rest api", "git", "docker", "postgresql"],
        "bonus": ["redis", "kafka", "microservices", "aws", "mongodb", "graphql"]
    },
    "devops engineer": {
        "required": ["docker", "kubernetes", "aws", "ci/cd", "linux", "terraform", "git", "bash"],
        "bonus": ["ansible", "prometheus", "grafana", "jenkins", "azure", "gcp"]
    }
}

print(f'Loaded {len(skills)} job roles:')
for role in skills:
    print(f'  - {role.title()} ({len(skills[role]["required"])} required, {len(skills[role]["bonus"])} bonus skills)')

## 4. TF-IDF Similarity Engine
Compute semantic similarity between resume text and role skill profile using TF-IDF vectorization and cosine similarity.

In [ ]:
def get_tfidf_score(resume_text, role_skills):
    """
    Compute TF-IDF cosine similarity between resume and role skill set.
    
    Args:
        resume_text (str): Raw resume text
        role_skills (list): List of skills for the target role
    
    Returns:
        float: Similarity score (0-100)
    """
    role_text = " ".join(role_skills)
    corpus = [resume_text.lower(), role_text.lower()]
    vectorizer = TfidfVectorizer()
    try:
        tfidf_matrix = vectorizer.fit_transform(corpus)
        similarity = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
        return round(float(similarity) * 100, 2)
    except:
        return 0.0

# Quick test
test_resume = "Experienced Python developer with skills in pandas, numpy, sql and data analysis"
test_skills = skills["data analyst"]["required"]
score = get_tfidf_score(test_resume, test_skills)
print(f'TF-IDF test score: {score}%')

## 5. Full Resume Analysis Pipeline

In [ ]:
def analyze_resume(resume_text, role):
    """
    Full resume analysis pipeline.
    
    Args:
        resume_text (str): Resume content
        role (str): Target job role
    
    Returns:
        dict: Analysis results including score, skills, suggestions
    """
    role = role.lower().strip()
    if role not in skills:
        return None

    required_skills = skills[role]["required"]
    bonus_skills    = skills[role]["bonus"]
    resume_lower    = resume_text.lower()

    # --- Keyword matching ---
    found_required   = [s for s in required_skills if s in resume_lower]
    missing_required = [s for s in required_skills if s not in resume_lower]
    found_bonus      = [s for s in bonus_skills    if s in resume_lower]

    # --- TF-IDF similarity ---
    all_role_skills = required_skills + bonus_skills
    tfidf_score     = get_tfidf_score(resume_text, all_role_skills)

    # --- Weighted final score (70% keyword + 30% TF-IDF) ---
    keyword_score = (len(found_required) / len(required_skills)) * 100
    final_score   = int((keyword_score * 0.7) + (tfidf_score * 0.3))
    final_score   = min(final_score, 100)

    # --- Seniority detection ---
    seniority     = "Mid-level"
    resume_words  = resume_lower.split()
    years_mentioned = [w for w in resume_words if w.isdigit() and 1 <= int(w) <= 20]
    if any(w in resume_lower for w in ["senior", "lead", "architect", "principal"]):
        seniority = "Senior"
    elif any(w in resume_lower for w in ["intern", "fresher", "entry", "junior", "graduate"]):
        seniority = "Entry-level"
    elif years_mentioned and int(years_mentioned[0]) >= 5:
        seniority = "Senior"

    # --- Suggestions ---
    suggestions = []
    if missing_required:
        suggestions.append(f"Add these key skills: {', '.join(missing_required[:3])}")
    if found_bonus:
        suggestions.append(f"Great bonus skills: {', '.join(found_bonus[:3])} — highlight these")
    elif bonus_skills:
        suggestions.append(f"Consider adding: {', '.join(bonus_skills[:3])} to stand out")
    if final_score >= 80:
        suggestions.append("Excellent match — tailor your summary for this role")
    elif final_score >= 55:
        suggestions.append("Good foundation — quantify your achievements")
    else:
        suggestions.append("Build projects using the missing skills")
    if len(resume_text.split()) < 150:
        suggestions.append("Resume is short — add more detail about projects and experience")
    if not any(c.isdigit() for c in resume_text):
        suggestions.append("Add measurable achievements (e.g. 'improved performance by 30%')")

    return {
        "score":          final_score,
        "keyword_score":  round(keyword_score),
        "tfidf_score":    round(tfidf_score),
        "found_required": found_required,
        "missing_required": missing_required,
        "found_bonus":    found_bonus,
        "seniority":      seniority,
        "suggestions":    suggestions,
        "role":           role
    }

print('analyze_resume() function defined successfully.')

## 6. Test with Sample Resumes

In [ ]:
# Sample resume 1 — Strong ML candidate
resume_ml = """
B.Tech Computer Science, 2024.
Skills: Python, TensorFlow, PyTorch, Scikit-learn, Pandas, NumPy, Data Preprocessing, Model Training.
Projects:
- Image classifier using CNN — achieved 94% accuracy on CIFAR-10
- NLP sentiment analysis model using BERT — F1 score 0.89
- Built end-to-end ML pipeline with feature engineering and cross-validation
Experience: ML Intern at XYZ Corp — improved model inference speed by 35%
"""

# Sample resume 2 — Junior software developer
resume_swe = """
Fresh graduate in Computer Science.
Skills: Python, Java, Git, SQL, Data Structures, Algorithms.
Projects:
- Voting system using Python and SQL
- Hand Cricket game using OpenCV
Languages: English, Telugu, Hindi
"""

result1 = analyze_resume(resume_ml,  "Machine Learning Engineer")
result2 = analyze_resume(resume_swe, "Software Developer")

print('=== Resume 1 — ML Engineer ===')
print(f'  Final Score    : {result1["score"]}/100')
print(f'  Keyword Match  : {result1["keyword_score"]}%')
print(f'  TF-IDF Score   : {result1["tfidf_score"]}%')
print(f'  Seniority      : {result1["seniority"]}')
print(f'  Found Skills   : {result1["found_required"]}')
print(f'  Missing Skills : {result1["missing_required"]}')

print()
print('=== Resume 2 — Software Developer ===')
print(f'  Final Score    : {result2["score"]}/100')
print(f'  Keyword Match  : {result2["keyword_score"]}%')
print(f'  TF-IDF Score   : {result2["tfidf_score"]}%')
print(f'  Seniority      : {result2["seniority"]}')
print(f'  Found Skills   : {result2["found_required"]}')
print(f'  Missing Skills : {result2["missing_required"]}')

## 7. Visualize Results

In [ ]:
# Bar chart — score breakdown comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('CareerAI — Resume Score Breakdown', fontsize=14, fontweight='bold', y=1.02)

for ax, result, title in zip(
    axes,
    [result1, result2],
    ['ML Engineer Resume', 'Software Developer Resume']
):
    categories = ['Final Score', 'Keyword Match', 'TF-IDF Score']
    values     = [result['score'], result['keyword_score'], result['tfidf_score']]
    colors     = ['#2563eb', '#16a34a', '#7c3aed']

    bars = ax.bar(categories, values, color=colors, width=0.5, edgecolor='white', linewidth=1.5)
    ax.set_ylim(0, 110)
    ax.set_title(title, fontsize=12, fontweight='bold', pad=12)
    ax.set_ylabel('Score (%)')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_facecolor('#f8faff')

    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                f'{val}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('score_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved as score_comparison.png')

In [ ]:
# Skill coverage — found vs missing
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
fig.suptitle('Skill Coverage Analysis', fontsize=14, fontweight='bold')

for ax, result, title in zip(
    axes,
    [result1, result2],
    ['ML Engineer', 'Software Developer']
):
    found   = len(result['found_required'])
    missing = len(result['missing_required'])
    bonus   = len(result['found_bonus'])

    sizes  = [found, missing, bonus] if bonus > 0 else [found, missing]
    labels = ['Found', 'Missing', 'Bonus'] if bonus > 0 else ['Found', 'Missing']
    colors = ['#16a34a', '#dc2626', '#2563eb'][:len(sizes)]

    wedges, texts, autotexts = ax.pie(
        sizes, labels=labels, colors=colors,
        autopct='%1.0f%%', startangle=90,
        wedgeprops={'edgecolor': 'white', 'linewidth': 2}
    )
    for at in autotexts:
        at.set_fontsize(11)
        at.set_fontweight('bold')

    ax.set_title(f'{title}\n(Seniority: {result["seniority"]})', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('skill_coverage.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved as skill_coverage.png')

In [ ]:
# All roles comparison — test same resume across all roles
roles   = list(skills.keys())
scores  = []

for role in roles:
    r = analyze_resume(resume_ml, role)
    scores.append(r['score'] if r else 0)

fig, ax = plt.subplots(figsize=(12, 5))
bar_colors = ['#2563eb' if s == max(scores) else '#93c5fd' for s in scores]
bars = ax.barh([r.title() for r in roles], scores, color=bar_colors, edgecolor='white', linewidth=1.5)

for bar, score in zip(bars, scores):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{score}%', va='center', fontsize=11, fontweight='bold')

ax.set_xlim(0, 115)
ax.set_xlabel('Match Score (%)', fontsize=11)
ax.set_title('Resume 1 — Match Score Across All Roles', fontsize=13, fontweight='bold', pad=14)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_facecolor('#f8faff')

best_role = roles[scores.index(max(scores))]
print(f'Best matched role: {best_role.title()} ({max(scores)}%)')

plt.tight_layout()
plt.savefig('role_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Try Your Own Resume

In [ ]:
# Paste your own resume here and choose a role
my_resume = """
Paste your resume text here...
"""

my_role = "Software Developer"  # Change this to your target role

result = analyze_resume(my_resume, my_role)

if result:
    print(f'Role          : {my_role}')
    print(f'Final Score   : {result["score"]}/100')
    print(f'Keyword Match : {result["keyword_score"]}%')
    print(f'TF-IDF Score  : {result["tfidf_score"]}%')
    print(f'Seniority     : {result["seniority"]}')
    print(f'Found Skills  : {", ".join(result["found_required"]) or "None"}')
    print(f'Missing Skills: {", ".join(result["missing_required"]) or "None"}')
    if result["found_bonus"]:
        print(f'Bonus Skills  : {", ".join(result["found_bonus"])}')
    print('\nSuggestions:')
    for s in result['suggestions']:
        print(f'  • {s}')
else:
    print(f'Role "{my_role}" not found. Available roles:')
    for r in skills: print(f'  - {r.title()}')

## 9. Run the Streamlit App

To launch the full web application:

In [ ]:
# Run this cell to launch the web app
# Then open http://localhost:8501 in your browser
!streamlit run app.py